In [ ]:
# ============================================================
# CELL 1 - Data harness + trustworthy local CV + flat baseline
# ============================================================
import os, glob, numpy as np, pandas as pd, warnings
warnings.filterwarnings("ignore")

CANDS = ["/kaggle/input/rogii-wellbore-geology-prediction",
         "/kaggle/input/competitions/rogii-wellbore-geology-prediction", "."]
DATA = next((p for p in CANDS if glob.glob(os.path.join(p,"**","*__horizontal_well.csv"),recursive=True)), ".")
TRAIN = os.path.join(DATA,"train")
if not glob.glob(os.path.join(TRAIN,"*__horizontal_well.csv")): TRAIN = DATA
print("DATA:", DATA, "| TRAIN:", TRAIN)

def pairs(folder):
    out=[]
    for hf in sorted(glob.glob(os.path.join(folder,"*__horizontal_well.csv"))):
        wid=os.path.basename(hf).split("__horizontal_well.csv")[0]
        tw=glob.glob(os.path.join(folder,f"{wid}__typewell*.csv"))
        if tw: out.append((wid,hf,tw[0]))
    return out

def load_h(p):
    d=pd.read_csv(p).sort_values("MD").reset_index(drop=True)
    if "TVT_input" not in d: d["TVT_input"]=np.nan
    return d
def load_t(p): return pd.read_csv(p).sort_values("TVT").reset_index(drop=True)
def ps_of(d):
    na=d["TVT_input"].isna().to_numpy(); return int(np.argmax(na)) if na.any() else len(d)

TR=[]
for wid,hf,tf in pairs(TRAIN):
    h=load_h(hf); ps=ps_of(h)
    if "TVT" in h and h["TVT"].notna().all() and 0<ps<len(h):
        TR.append((wid,hf,tf,ps,len(h)))
print(f"usable train wells: {len(TR)}")



In [ ]:
# ============================================================
# CELL 4 - Leak-free particle filter, scored on OUR harness
# ============================================================
import numpy as np, pandas as pd, time
try:
    from numba import njit
except ImportError:
    def njit(**k):
        return lambda f: f

@njit(cache=False)
def _pf_core(md_v, z_v, gr_v, gg, vmin, gstep, gs, ls, ir, N,
             MOM, VN, PN, RP, RR, RESAMP, spr, seed):
    np.random.seed(seed)
    n = len(md_v)
    out = np.empty(n)
    pos = np.empty(N); rate = np.empty(N); w = np.ones(N)/N
    for j in range(N):
        pos[j] = ls + spr*np.random.randn()
        rate[j] = ir + 0.01*np.random.randn()
    prev = md_v[0]-1.0
    ng = len(gg)-1
    for i in range(n):
        dm = md_v[i]-prev
        if dm < 1.0: dm = 1.0
        for j in range(N):
            rate[j] = MOM*rate[j] + VN*np.random.randn()
            pos[j] += rate[j]*dm + PN*np.random.randn()
        for j in range(N):
            t = (pos[j]-z_v[i]-vmin)/gstep
            k = int(t)
            if k < 0: eg = gg[0]
            elif k >= ng: eg = gg[ng]
            else:
                f = t-k; eg = gg[k]*(1.0-f)+gg[k+1]*f
            d = (gr_v[i]-eg)/gs; dd = d*d
            if dd > 600.0: dd = 600.0
            lk = np.exp(-0.5*dd)
            if lk < 1e-300: lk = 1e-300
            w[j] = w[j]*lk
        s = 0.0
        for j in range(N): s += w[j]
        if s > 0.0:
            for j in range(N): w[j] /= s
        else:
            for j in range(N): w[j] = 1.0/N
        ne = 0.0
        for j in range(N): ne += w[j]*w[j]
        if 1.0/ne < RESAMP*N:
            cum = np.empty(N); c = 0.0
            for j in range(N): c += w[j]; cum[j] = c
            u0 = np.random.uniform(0.0, 1.0/N); ci = 0
            npos = np.empty(N); nrate = np.empty(N)
            for j in range(N):
                u = u0+j/N
                while ci < N-1 and cum[ci] < u: ci += 1
                npos[j] = pos[ci]+RP*np.random.randn()
                nrate[j] = rate[ci]+RR*np.random.randn()
            for j in range(N):
                pos[j] = npos[j]; rate[j] = nrate[j]; w[j] = 1.0/N
        e = 0.0
        for j in range(N): e += w[j]*(pos[j]-z_v[i])
        out[i] = e; prev = md_v[i]
    return out

def pf_predict(h, t, ps, n_part=400, n_seeds=8, gs_scale=8.0):
    tw = t.sort_values("TVT")
    tvt_g = tw.TVT.values.astype(float)
    gr_g  = tw.GR.fillna(tw.GR.mean()).values.astype(float)
    vmin = tvt_g[0]; gstep = np.median(np.diff(tvt_g))
    grid = np.arange(vmin, tvt_g[-1]+gstep, gstep)
    gg = np.interp(grid, tvt_g, gr_g)

    kn = h.iloc[:ps]; ev = h.iloc[ps:]
    tvt_k = kn.TVT_input.values.astype(float)
    twk = np.interp(tvt_k, tvt_g, gr_g)
    gs = float(np.clip(np.nanstd(kn.GR.fillna(0).values-twk), 10., 60.))/8.0*gs_scale

    dt = np.diff(tvt_k[-30:]); dz = np.diff(kn.Z.values[-30:]); dm = np.diff(kn.MD.values[-30:])
    m = dm > 0
    ir = float(np.median((dt+dz)[m]/dm[m])) if m.sum() >= 3 else 0.0

    ls = float(tvt_k[-1]+kn.Z.values[-1])
    gri = h.GR.interpolate(limit_direction="both").fillna(float(gr_g.mean()))
    md_v = ev.MD.values.astype(float); z_v = ev.Z.values.astype(float)
    gr_v = gri.values.astype(float)[ps:]

    acc = np.zeros(len(md_v))
    for s in range(n_seeds):
        acc += _pf_core(md_v, z_v, gr_v, gg, vmin, gstep, gs, ls, ir, n_part,
                        0.993, 0.005, 0.01, 0.2, 0.003, 0.5, 4.5, 42+s)
    return acc/n_seeds



In [ ]:
# ============================================================
# CELL 6 - Spatial formation imputation (THE lever). Run after CELL 1.
# ============================================================
import numpy as np, pandas as pd, time
from scipy.spatial import cKDTree
FORM = ['ANCC','ASTNU','ASTNL','EGFDU','EGFDL','BUDA']

# ---- build the field point cloud from ALL train wells ----
PT, TREE = {}, {}
t0 = time.time()
for wid, hf, tf, ps, ln in TR:
    h = load_h(hf)
    if not set(FORM).issubset(h.columns): continue
    sl = slice(0, len(h), 20)                      # decimate: surfaces are smooth
    d = {c: h[c].values[sl].astype(float) for c in FORM}
    ok = np.all([np.isfinite(d[c]) for c in FORM], axis=0)
    if ok.sum() < 5: continue
    x = h.X.values[sl][ok]; y = h.Y.values[sl][ok]
    PT[wid] = (x, y, {c: d[c][ok] for c in FORM})
    TREE[wid] = cKDTree(np.c_[x, y])
print(f"field cloud: {len(PT)} wells, {sum(len(v[0]) for v in PT.values())} pts [{time.time()-t0:.0f}s]")

WLIST = list(PT.keys())
HEEL = np.array([[PT[w][0].mean(), PT[w][1].mean()] for w in WLIST])
HTREE = cKDTree(HEEL)

def impute_perwell(target, xq, yq, kw=8, cand=25, ridge=1e-4):
    """Nearest point from each of the kw nearest WELLS -> conditioned local plane."""
    cx, cy = np.mean(xq), np.mean(yq)
    _, ci = HTREE.query([cx, cy], k=min(cand, len(WLIST)))
    others = [WLIST[j] for j in np.atleast_1d(ci) if WLIST[j] != target]
    if len(others) < 4: return None, None
    Q = np.c_[xq, yq]
    dd = np.empty((len(others), len(xq))); ii = np.empty((len(others), len(xq)), int)
    for j, w in enumerate(others):
        d, i = TREE[w].query(Q, k=1); dd[j] = d; ii[j] = i
    order = np.argsort(dd, axis=0)[:kw]
    out = {c: np.full(len(xq), np.nan) for c in FORM}; nn = np.empty(len(xq))
    for q in range(len(xq)):
        js = order[:, q]
        ws = [others[j] for j in js]; ix = [ii[j, q] for j in js]
        px = np.array([PT[w][0][k] for w, k in zip(ws, ix)])
        py = np.array([PT[w][1][k] for w, k in zip(ws, ix)])
        nn[q] = dd[js, q].min()
        A = np.c_[px-xq[q], py-yq[q], np.ones(len(px))]
        AtA = A.T@A + ridge*np.eye(3)
        for c in FORM:
            v = np.array([PT[w][2][c][k] for w, k in zip(ws, ix)])
            m = np.isfinite(v)
            if m.sum() < 3: out[c][q] = np.nanmean(v) if m.any() else np.nan; continue
            try:
                sol = np.linalg.solve(A[m].T@A[m] + ridge*np.eye(3), A[m].T@v[m])
                out[c][q] = sol[2]
            except Exception:
                out[c][q] = v[m].mean()
    return out, nn



In [ ]:
# ============================================================
# CELL 22 - Detrended IDW (removes the dip before interpolating)
# ============================================================
import numpy as np, random, time
from scipy.spatial import cKDTree

# --- 1. fit a global trend per formation from ALL train wells ---
TREND = {}
AX, AY = [], []
for w,(x,y,d) in PT.items(): AX.append(x); AY.append(y)
AX = np.concatenate(AX); AY = np.concatenate(AY)
x0, y0 = AX.mean(), AY.mean()
for c in FORM:
    Xs, Ys, Vs = [], [], []
    for w,(x,y,d) in PT.items():
        m = np.isfinite(d[c]); Xs.append(x[m]); Ys.append(y[m]); Vs.append(d[c][m])
    X = np.concatenate(Xs)-x0; Y = np.concatenate(Ys)-y0; V = np.concatenate(Vs)
    A = np.c_[X, Y, np.ones(len(X))]
    TREND[c] = np.linalg.lstsq(A, V, rcond=None)[0]
    print(f"{c:7} gx={TREND[c][0]:+.5f} gy={TREND[c][1]:+.5f} |g|={np.hypot(*TREND[c][:2]):.4f}")

def trend_at(c, x, y):
    g = TREND[c]; return g[0]*(x-x0) + g[1]*(y-y0) + g[2]

# --- 2. precompute detrended residuals (the surface, minus its dip) ---
RES = {w: {c: PT[w][2][c] - trend_at(c, PT[w][0], PT[w][1]) for c in FORM} for w in PT}

def impute_dt(target, xq, yq, kw=8, cand=25, ls=600.0, use_trend=True):
    cx, cy = np.mean(xq), np.mean(yq)
    _, ci = HTREE.query([cx, cy], k=min(cand, len(WLIST)))
    others = [WLIST[j] for j in np.atleast_1d(ci) if WLIST[j] != target]
    if len(others) < 3: return None, None
    Q = np.c_[xq, yq]
    dd = np.empty((len(others), len(xq))); ii = np.empty((len(others), len(xq)), int)
    for j, w in enumerate(others):
        d, i = TREE[w].query(Q, k=1); dd[j] = d; ii[j] = i
    order = np.argsort(dd, axis=0)[:kw]
    out = {c: np.empty(len(xq)) for c in FORM}; nn = np.empty(len(xq))
    for q in range(len(xq)):
        js = order[:, q]
        ws = [others[j] for j in js]; ix = [ii[j, q] for j in js]
        dq = dd[js, q]; nn[q] = dq.min()
        wt = 1.0/(dq**2 + ls**2)
        for c in FORM:
            src = RES if use_trend else PT
            v = np.array([(RES[w][c][k] if use_trend else PT[w][2][c][k])
                          for w, k in zip(ws, ix)])
            m = np.isfinite(v)
            val = float((v[m]*wt[m]).sum()/wt[m].sum()) if m.any() else np.nan
            out[c][q] = val + (trend_at(c, xq[q], yq[q]) if use_trend else 0.0)
    return out, nn



In [ ]:
# ============================================================
# CELL 23 - Curvature: quadratic + LOCAL trend
# ============================================================
import numpy as np, random, time

# ---- (a) is the "6 surfaces" really 1 surface? ----
ref = 'EGFDU'
for c in FORM:
    th = np.concatenate([PT[w][2][c] - PT[w][2][ref] for w in PT])
    th = th[np.isfinite(th)]
    print(f"{c:7} thickness vs {ref}: mean={th.mean():8.2f}  std={th.std():.3f}")

# ---- (b) QUADRATIC global trend ----
def design(x, y):
    dx, dy = x-x0, y-y0
    return np.c_[dx, dy, dx*dx, dy*dy, dx*dy, np.ones(len(x))]

TQ = {}
for c in FORM:
    Xs, Ys, Vs = [], [], []
    for w,(x,y,d) in PT.items():
        m = np.isfinite(d[c]); Xs.append(x[m]); Ys.append(y[m]); Vs.append(d[c][m])
    X = np.concatenate(Xs); Y = np.concatenate(Ys); V = np.concatenate(Vs)
    A = design(X, Y)
    TQ[c] = np.linalg.lstsq(A, V, rcond=None)[0]
    print(f"{c:7} quad-trend residual std = {np.std(V - A@TQ[c]):7.2f} ft "
          f"(linear was {np.std(V - np.c_[X-x0,Y-y0,np.ones(len(X))]@TREND[c]):.2f})")

def trend_q(c, x, y): return design(np.atleast_1d(x), np.atleast_1d(y)) @ TQ[c]
RESQ = {w: {c: PT[w][2][c] - trend_q(c, PT[w][0], PT[w][1]) for c in FORM} for w in PT}

# ---- (c) LOCAL trend: plane from ALL points of the CAND nearest wells ----
def impute_loc(target, xq, yq, kw=8, cand=25, ls=600.0, ltrend=True, lcand=20):
    cx, cy = np.mean(xq), np.mean(yq)
    _, ci = HTREE.query([cx, cy], k=min(cand, len(WLIST)))
    others = [WLIST[j] for j in np.atleast_1d(ci) if WLIST[j] != target]
    if len(others) < 3: return None, None
    LT = {}
    if ltrend:
        src = others[:lcand]
        px = np.concatenate([PT[w][0] for w in src]); py = np.concatenate([PT[w][1] for w in src])
        for c in FORM:
            v = np.concatenate([PT[w][2][c] for w in src]); m = np.isfinite(v)
            if m.sum() < 50: LT[c] = None; continue
            A = np.c_[px[m]-cx, py[m]-cy, np.ones(m.sum())]
            LT[c] = np.linalg.lstsq(A, v[m], rcond=None)[0]
    def tr(c, x, y):
        if not ltrend or LT.get(c) is None: return trend_at(c, x, y)
        g = LT[c]; return g[0]*(x-cx) + g[1]*(y-cy) + g[2]
    Q = np.c_[xq, yq]
    dd = np.empty((len(others), len(xq))); ii = np.empty((len(others), len(xq)), int)
    for j, w in enumerate(others):
        dv, iv = TREE[w].query(Q, k=1); dd[j] = dv; ii[j] = iv
    order = np.argsort(dd, axis=0)[:kw]
    out = {c: np.empty(len(xq)) for c in FORM}; nn = np.empty(len(xq))
    for q in range(len(xq)):
        js = order[:, q]; ws = [others[j] for j in js]; ix = [ii[j, q] for j in js]
        dq = dd[js, q]; nn[q] = dq.min(); wt = 1.0/(dq**2 + ls**2)
        for c in FORM:
            v = np.array([PT[w][2][c][k] - tr(c, PT[w][0][k], PT[w][1][k])
                          for w, k in zip(ws, ix)])
            m = np.isfinite(v)
            r = float((v[m]*wt[m]).sum()/wt[m].sum()) if m.any() else 0.0
            out[c][q] = r + tr(c, xq[q], yq[q])
    return out, nn



In [ ]:
# Inference-only controls; training CACHE is packaged into TRUST weights.
DATUM_WINDOWS=[("full",None),("late2",1500),("late",500),("last",150)]
PR_WIN=500


In [ ]:
# Per-well TRUST is loaded from the private legal model package below.


In [ ]:
"""
A wider feature set feeding a boosted ensemble, floored on the blend.
This version also exposes each of the six formation markers individually rather than
only their reliability-weighted average: per marker we give the model its own TVT
prediction (tvtF), the shift between its full-prefix and late-prefix datum (bw, a
per-marker drift signal), and its heel reliability (frm). This lets the model learn
which markers to trust per well instead of committing to one aggregate.
This extends the previous version with a second datum-scan curve taken around the
imputation/production track (so ambiguity is measured against both independent
anchors), a rolling gamma-ray energy term, additional lag correlations, and
nonlinear position terms. Same out-of-well GroupKFold and ridge-floor safety.
 
Earlier ensembles saw only a handful of features that were largely recombinations of the two
existing signals, so a boosted model found little to add. Here we widen the description of
each evaluation row with families that carry genuinely new information about the gamma-ray log
and the alignment quality, then let the ensemble weigh them. All of these are computed from
each well's own gamma ray, geometry, and typewell, so nothing depends on the test labels.
 
New feature families.
  Datum-scan curve. For a grid of vertical shifts we slide the particle-filter track up and
  down and measure the gamma-ray misfit against the typewell, J(shift) = mean_i
  (GR_cal_i - G_tw(PF_i + shift))^2. The depth of the minimum measures residual datum error,
  and the shape of the curve (its sharpness, and whether it has a second basin) exposes the
  low/high ambiguity that a single number cannot. We expose the curve values, its minimum,
  the shift at the minimum, and its curvature.
 
  Heel calibration. Gamma ray differs in gain and offset between the horizontal well and the
  typewell. On the known heel we fit GR = a * G_tw(TVT) + b and carry (a, b) and the fit
  residual, then use the calibrated gamma ray everywhere downstream.
 
  Multi-scale gamma-ray character. Rolling mean and standard deviation at windows 5, 21, 51,
  101, the first and second derivatives, an envelope, and lead/lag autocorrelations. These
  summarize the local log texture that a point-wise filter does not represent.
 
  Slope and geometry. The heel TVT slope, the local apparent dip dZ/dMD, the vertical climb,
  normalized measured depth, and well-summary terms.
 
Model and safety. Two LightGBM configurations and a CatBoost model predict the correction
dTVT with 5-fold GroupKFold. A non-negative ridge over [fixed_blend, boosted_predictions]
guarantees the result cannot fall below the fixed blend on the out-of-well folds. The path is
finished with the robust degree-4 projection.
"""

In [ ]:
import numpy as np, pandas as pd, os, glob, time, random, pickle, warnings
warnings.filterwarnings("ignore")
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import Ridge
import lightgbm as lgb
try:
    from catboost import CatBoostRegressor; HAVE_CB=True
except Exception:
    HAVE_CB=False
try:
    from numba import njit
except ImportError:
    def njit(**k): return lambda f: f
 
PF_TRAIN_WELLS=765; PF_SEEDS,PF_PART=24,300; TRAIN_STRIDE=8
W_PROD,W_PF=0.55,0.45; TAU,A_TRUST=85.0,0.70; USE_CB=True
PROJ_DEG,PROJ_BLEND=4,1.0; PF_CACHE="pf_train_cache.pkl"
SHIFTS=[-30,-15,-8,-4,-2,0,2,4,8,15,30]
 
@njit(cache=False, nogil=True)
def _pf(md_v,z_v,gr_v,gg,vmin,step,gs,ls,ir,N,n_seeds,MOM,VN,PN,RP,RR,RESAMP,init_spr):
    n=len(md_v); preds=np.empty((n_seeds,n)); liks=np.empty(n_seeds); ng=len(gg)-1
    for s in range(n_seeds):
        np.random.seed(s)
        pos=np.empty(N); rate=np.empty(N); w=np.ones(N)/N
        for j in range(N): pos[j]=ls+init_spr*np.random.randn(); rate[j]=ir+0.01*np.random.randn()
        ll=0.0; prev=md_v[0]-1.0
        for i in range(n):
            dm=md_v[i]-prev
            if dm<1.0: dm=1.0
            for j in range(N):
                rate[j]=MOM*rate[j]+VN*np.random.randn(); pos[j]+=rate[j]*dm+PN*np.random.randn()
            avg=0.0
            for j in range(N):
                t=(pos[j]-z_v[i]-vmin)/step; k=int(t)
                if k<0: eg=gg[0]
                elif k>=ng: eg=gg[ng]
                else: f=t-k; eg=gg[k]*(1.0-f)+gg[k+1]*f
                dd=((gr_v[i]-eg)/gs)**2
                if dd>600.0: dd=600.0
                lk=np.exp(-0.5*dd)
                if lk<1e-300: lk=1e-300
                avg+=w[j]*lk; w[j]=w[j]*lk
            if avg<1e-300: avg=1e-300
            ll+=np.log(avg)
            ws=0.0
            for j in range(N): ws+=w[j]
            if ws>0.0:
                for j in range(N): w[j]/=ws
            else:
                for j in range(N): w[j]=1.0/N
            neff=0.0
            for j in range(N): neff+=w[j]*w[j]
            neff=1.0/neff
            if neff<RESAMP*N:
                cum=np.empty(N); c=0.0
                for j in range(N): c+=w[j]; cum[j]=c
                u0=np.random.uniform(0.,1.0/N); npx=np.empty(N); nrt=np.empty(N); ci=0
                for j in range(N):
                    u=u0+j/N
                    while ci<N-1 and cum[ci]<u: ci+=1
                    npx[j]=pos[ci]+RP*np.random.randn(); nrt[j]=rate[ci]+RR*np.random.randn()
                for j in range(N): pos[j]=npx[j]; rate[j]=nrt[j]; w[j]=1.0/N
            est=0.0
            for j in range(N): est+=w[j]*(pos[j]-z_v[i])
            preds[s,i]=est; prev=md_v[i]
        liks[s]=ll
    return preds,liks
 
def pf_scales(h,t,ps,n_part=PF_PART,n_seeds=PF_SEEDS):
    tw=t.sort_values("TVT"); tv=tw.TVT.values.astype(float); tg=tw.GR.fillna(tw.GR.mean()).values.astype(float)
    kn=h.iloc[:ps]; ev=h.iloc[ps:]
    if len(ev)==0 or len(tv)<3: return None
    ls=float(kn.TVT_input.values[-1]+kn.Z.values[-1]); twk=np.interp(kn.TVT_input.values,tv,tg)
    gs=float(np.clip(np.nanstd(kn.GR.fillna(0).values-twk),10.,60.))*1.3
    tail=kn.tail(30); dt=np.diff(tail.TVT_input.values); dz=np.diff(tail.Z.values); dm=np.diff(tail.MD.values); m=dm>0
    ir=float(np.median((dt+dz)[m]/dm[m])) if m.sum()>=3 else 0.0
    vmin=tv[0]; step=float(np.median(np.diff(tv))) or 1.0
    grid=np.arange(vmin,tv[-1]+step,step); gg=np.interp(grid,tv,tg).astype(float)
    gr_v=h.GR.interpolate(limit_direction="both").fillna(tg.mean()).values.astype(float)[ps:]
    preds,liks=_pf(ev.MD.values.astype(float),ev.Z.values.astype(float),gr_v,gg,vmin,step,gs,ls,ir,
                   n_part,n_seeds,0.998,0.002,0.005,0.1,0.001,0.5,4.5)
    ln=liks-liks.max(); out={}
    for tag,sc in [('s3',3.),('s5',5.),('s8',8.)]:
        wt=np.exp(ln/sc); wt/=wt.sum(); out[tag]=(wt[:,None]*preds).sum(0).astype(np.float32)
    out['pstd']=preds.std(0).astype(np.float32)
    return out
 
def warmup(md,tau=TAU): return 1-np.exp(-np.maximum(md,0)/tau)
def prod_trust(w): return A_TRUST*float(WH.get(w,SH0))+(1-A_TRUST)*SH0
def agg(d,pw=2.0):
    cs=[c for c in d['P'] if c in d['PR']]
    if not cs: return d['flat'].astype(float).copy(), np.zeros(len(d['flat'])), 0.0
    M=np.stack([d['P'][c] for c in cs]); w=np.array([1.0/(d['PR'][c]**pw+1e-6) for c in cs]); w/=w.sum()
    return (M*w[:,None]).sum(0), M.std(0), float(np.ptp([d['PR'][c] for c in cs]))
def robfit(s,y,deg=PROJ_DEG):
    if len(s)<deg+2: return y.copy()
    c=np.polyfit(s,y,deg)
    for _ in range(4):
        r=y-np.polyval(c,s); sc=np.median(np.abs(r))*1.4826+1e-6
        c=np.polyfit(s,y,deg,w=1.0/(1.0+(r/(2.0*sc))**2))
    return np.polyval(c,s)
def apply_proj(tvt,Z,s,anchor):
    D=robfit(s,(tvt+Z)-anchor,PROJ_DEG); return (1-PROJ_BLEND)*tvt+PROJ_BLEND*((anchor+D)-Z)
def rollms(a,w):
    s=pd.Series(a); return (s.rolling(w,center=True,min_periods=1).mean().values,
                            s.rolling(w,center=True,min_periods=1).std().fillna(0).values)
def lagc(a,k):
    if k>=len(a): return 0.0
    x,y=a[:-k],a[k:]
    if len(x)<3 or np.std(x)<1e-6 or np.std(y)<1e-6: return 0.0
    return float(np.corrcoef(x,y)[0,1])
 
def extra_features(h,ps,t,pfmean,prod_track):
    """Datum-scan curve, heel calibration, GR signal, slope. pfmean = per-row PF TVT track."""
    kn=h.iloc[:ps]; ev=h.iloc[ps:]
    GR=h.GR.interpolate(limit_direction="both").fillna(0).values.astype(float)
    gr_ev=GR[ps:]; tv=t.sort_values("TVT").TVT.values.astype(float)
    tg=t.sort_values("TVT").GR.fillna(t.GR.mean()).values.astype(float)
    tvt_k=kn.TVT_input.values.astype(float); gr_k=GR[:ps]; gtwk=np.interp(tvt_k,tv,tg)
    A=np.c_[gtwk,np.ones(len(gtwk))]
    try: c=np.linalg.lstsq(A,gr_k,rcond=None)[0]; cal_a,cal_b=float(c[0]),float(c[1])
    except Exception: cal_a,cal_b=1.0,0.0
    pfx_rmse=float(np.sqrt(np.mean((A@[cal_a,cal_b]-gr_k)**2)))
    gr_cal=(gr_ev-cal_b)/(cal_a if abs(cal_a)>1e-3 else 1.0)
    lo,hi=tv[0],tv[-1]
    scan={}
    for s_ in SHIFTS:
        q=np.clip(pfmean+s_,lo,hi); scan[f"tda{s_}"]=float(np.mean((gr_cal-np.interp(q,tv,tg))**2))
    sv=np.array([scan[f"tda{s}"] for s in SHIFTS]); ai=int(np.argmin(sv))
    tda_min=float(sv.min()); tda_argmin=float(SHIFTS[ai])
    tda_curv=float(sv[max(ai-1,0)]+sv[min(ai+1,len(sv)-1)]-2*sv[ai])
    slp_all=float(np.polyfit(kn.MD.values[-min(len(kn),500):],tvt_k[-min(len(kn),500):],1)[0]) if len(kn)>2 else 0.0
    dzdmd=float(np.polyfit(kn.MD.values[-30:],kn.Z.values[-30:],1)[0]) if len(kn)>=3 else 0.0
    gr_d1=np.gradient(gr_ev); gr_d2=np.gradient(gr_d1)
    base21=pd.Series(gr_ev).rolling(21,center=True,min_periods=1).mean().values
    gr_env=pd.Series(np.abs(gr_ev-base21)).rolling(21,center=True,min_periods=1).max().values
    gr_nrg=pd.Series(gr_ev**2).rolling(21,center=True,min_periods=1).mean().values
    per=dict(gr_d1=gr_d1,gr_d2=gr_d2,gr_env=gr_env,gr_nrg=gr_nrg)
    for w in (5,21,51,101):
        m,s=rollms(gr_ev,w); per[f"grm{w}"]=m; per[f"grs{w}"]=s
    scanp={}
    for s_ in [-15,-4,0,4,15]:
        q=np.clip(prod_track+s_,lo,hi); scanp[f'tdp{s_}']=float(np.mean((gr_cal-np.interp(q,tv,tg))**2))
    slp_50=float(np.polyfit(kn.MD.values[-min(len(kn),50):],tvt_k[-min(len(kn),50):],1)[0]) if len(kn)>2 else 0.0
    ktvt_range=float(np.ptp(tvt_k)) if len(tvt_k) else 0.0
    ws=dict(cal_a=cal_a,cal_b=cal_b,pfx_rmse=pfx_rmse,slp_all=slp_all,slp_50=slp_50,ktvt_range=ktvt_range,dzdmd=dzdmd,
            glag1=lagc(gr_ev,1),glag5=lagc(gr_ev,5),glag15=lagc(gr_ev,15),glag30=lagc(gr_ev,30),tda_min=tda_min,tda_argmin=tda_argmin,
            tda_curv=tda_curv,known_len=float(ps),eval_len=float(len(ev)),
            ktvt_std=float(np.std(tvt_k)),**scan,**scanp)
    return per,ws
 
def marker_feats(d, flat):
    out={}
    for c in FORM:
        kf=f'{c}_full'; kl=f'{c}_late'
        if kf in d['P']:
            out[f'tvtF_{c}']=d['P'][kf].astype(float)-flat
            out[f'frm_{c}']=float(d['PR'].get(kf,np.nan))
            out[f'bw_{c}']=(d['P'][kl].astype(float)-d['P'][kf].astype(float)) if kl in d['P'] else np.zeros(len(flat))
        else:
            out[f'tvtF_{c}']=np.full(len(flat),np.nan)
            out[f'frm_{c}']=np.nan
            out[f'bw_{c}']=np.zeros(len(flat))
    return out
MARK_F=[f'tvtF_{c}' for c in FORM]+[f'bw_{c}' for c in FORM]+[f'frm_{c}' for c in FORM]
HFm={wid:(hf,tf,ps) for wid,hf,tf,ps,ln in TR}

SCAN_F=[f"tda{s}" for s in SHIFTS]
PER_F=['gr_d1','gr_d2','gr_env','gr_nrg','grm5','grs5','grm21','grs21','grm51','grs51','grm101','grs101']
WS_F=['cal_a','cal_b','pfx_rmse','slp_all','slp_50','ktvt_range','dzdmd','glag1','glag5','glag15','glag30',
      'tda_min','tda_argmin','tda_curv','known_len','eval_len','ktvt_std']+SCAN_F+['tdp-15','tdp-4','tdp0','tdp4','tdp15']
CORE=['prod_d','imp_raw_d','imp_spread','disagree','frac2','sqrt_frac','pf_s3_d','pf_s5_d','pf_s8_d','pf_mean_d','pf_std',
      'pf_gap','blend_d','s','md_since','dZ','Z','dip','X','Y','z_span','GR','GR_dev','nn',
      'pr_min','pr_med','pr_max','pr_std','n_form']
FEATS_R=CORE+PER_F+WS_F+MARK_F
 

# ---- inference ----

from pathlib import Path as _Path
import json as _json
import lightgbm as _lgb
import xgboost as _xgb
_har_candidates=[
 _Path("/kaggle/input/datasets/boltuzamaki/rogii-harshini-fast-models"),
 _Path("/kaggle/input/rogii-harshini-fast-models")]
_har_pkg=next((p for p in _har_candidates if p.exists()),None)
if _har_pkg is None: raise RuntimeError("Harshini fast model package missing")
_har_manifest=_json.loads((_har_pkg/"manifest.json").read_text())
if set(FEATS_R) != set(_har_manifest["features"]) or len(FEATS_R)!=92:
    raise RuntimeError("Harshini 92-feature schema differs from trained manifest")
_HLGB=_lgb.Booster(model_file=str(_har_pkg/"harshini_fast_lgb.txt"))
_HXGB=_xgb.XGBRegressor()
_HXGB.load_model(_har_pkg/"harshini_fast_xgb.json")
_HXGB.set_params(device="cpu",n_jobs=-1)
_tcfg=_har_manifest["trust_model"]
SH0=float(_tcfg["SH0"]); CL0=float(_tcfg["CL0"])
A_TRUST=float(_tcfg["A_TRUST"]); FEATS=list(_tcfg["features"])
TRUST=_lgb.Booster(model_file=str(_har_pkg/"harshini_trust_lgb.txt"))
def surf_cache(h,ps):
    fev,dnn=impute_loc(None,h.X.values[ps:],h.Y.values[ps:],ls=600.0,ltrend=True,lcand=40)
    fkn,_=impute_loc(None,h.X.values[:ps],h.Y.values[:ps],ls=600.0,ltrend=True,lcand=40)
    zev=h.Z.values[ps:]; zkn=h.Z.values[:ps]; ktvt=h.TVT_input.values[:ps]
    tvt0=float(h.TVT_input.values[ps-1]); nw=slice(-min(PR_WIN,ps),None); P,PR={},{}
    if fev is not None:
        for c in FORM:
            if not (np.isfinite(fev[c]).all() and np.isfinite(fkn[c]).all()): continue
            r=ktvt+zkn-fkn[c]
            if not np.isfinite(r).any(): continue
            for tag,win in DATUM_WINDOWS:
                sel=slice(None) if win is None else slice(-min(win,ps),None); rr=r[sel]
                if not np.isfinite(rr).any(): continue
                b=float(np.nanmedian(rr)); p=-zev+fev[c]+b
                if not np.isfinite(p).all(): continue
                P[f"{c}_{tag}"]=p.astype(np.float32)
                PR[f"{c}_{tag}"]=float(np.sqrt(np.nanmean((ktvt[nw]-(-zkn[nw]+fkn[c][nw]+b))**2)))
    return dict(P=P,PR=PR,flat=np.full(len(zev),tvt0,np.float32),zev=zev.astype(np.float32),
                nn=float(np.nanmean(dnn)) if fev is not None else 9e3,n=len(zev)),tvt0
TEST=os.path.join(DATA,"test")
if not glob.glob(os.path.join(TEST,"*__horizontal_well.csv")): TEST=DATA
rows=[]
for wid,hf,tf in pairs(TEST):
    h=load_h(hf); t=load_t(tf); ps=ps_of(h)
    if ps>=len(h) or ps<1: continue
    d,tvt0=surf_cache(h,ps); flat=d['flat'].astype(float)
    imp_raw,disagree,imp_spread=agg(d); imp_raw_d=imp_raw-flat
    prs=np.array(list(d['PR'].values())) if d['PR'] else np.array([np.nan])
    tfr=pd.DataFrame([dict(pr_min=prs.min(),pr_med=np.median(prs),pr_max=prs.max(),pr_std=prs.std(),
        nn=d['nn'],dl_absmean=np.abs(np.clip(imp_raw_d,-CL0,CL0)).mean(),dl_absmax=np.abs(np.clip(imp_raw_d,-CL0,CL0)).max(),
        dl_std=np.clip(imp_raw_d,-CL0,CL0).std(),dl_end=abs(np.clip(imp_raw_d,-CL0,CL0)[-1]) if len(imp_raw_d) else 0.0,
        dl_slope=np.polyfit(np.linspace(0,1,len(imp_raw_d)),np.clip(imp_raw_d,-CL0,CL0),1)[0] if len(imp_raw_d)>1 else 0.0,
        disagree=disagree.mean(),disagree_max=disagree.max(),n_form=len(d['P']))])[FEATS]
    trust=A_TRUST*float(np.clip(TRUST.predict(tfr)[0],0,1.2))+(1-A_TRUST)*SH0
    prod_d=trust*np.clip(imp_raw_d,-CL0,CL0)
    pf=pf_scales(h,t,ps,n_part=400,n_seeds=96)
    MD=h.MD.values[ps:].astype(float); md0=float(h.MD.values[ps-1]); Z=d['zev'].astype(float); z0=float(h.Z.values[ps-1])
    GR=h.GR.values[ps:].astype(float); grp=float(np.nanmean(h.GR.values[:ps])) if np.isfinite(h.GR.values[:ps]).any() else 0.0
    s=(MD-md0)/max(MD[-1]-md0,1e-6); anchor=float(tvt0)+z0
    if pf is not None:
        s3d=pf['s3']-flat; s5d=pf['s5']-flat; s8d=pf['s8']-flat; pfmean=(pf['s3']+pf['s5']+pf['s8'])/3.0; pstd=pf['pstd']
    else:
        s3d=s5d=s8d=np.zeros(len(flat)); pfmean=flat.copy(); pstd=np.zeros(len(flat))
    pfmean_d=pfmean-flat; blend_d=W_PROD*prod_d+W_PF*pfmean_d
    dip=np.gradient(Z)/np.clip(np.gradient(MD),1e-6,None)
    per,ws=extra_features(h,ps,t,pfmean,flat+prod_d)
    row=dict(prod_d=prod_d,imp_raw_d=imp_raw_d,imp_spread=imp_spread,disagree=disagree,pf_s3_d=s3d,
        pf_s5_d=s5d,pf_s8_d=s8d,pf_mean_d=pfmean_d,pf_std=pstd,pf_gap=pfmean_d-prod_d,blend_d=blend_d,
        s=s,md_since=MD-md0,dZ=Z-z0,Z=Z,dip=dip,X=float(h.X.mean()),Y=float(h.Y.mean()),
        z_span=float(Z.max()-Z.min()),GR=np.nan_to_num(GR,nan=grp),GR_dev=np.nan_to_num(GR,nan=grp)-grp,
        nn=d['nn'],pr_min=prs.min(),pr_med=np.median(prs),pr_max=prs.max(),pr_std=prs.std(),n_form=len(d['P']),
        frac2=s**2,sqrt_frac=np.sqrt(s))
    row.update(per)
    for k,v in ws.items(): row[k]=v
    for k,v in marker_feats(d,flat).items(): row[k]=v
    F=pd.DataFrame(row)[_har_manifest["features"]].values.astype(np.float32)
    _wu=warmup(MD-md0)
    _plgb=_wu*_HLGB.predict(F)
    _pxgb=_wu*_HXGB.predict(F)
    _pphy=_wu*blend_d
    for j,i in enumerate(range(ps,len(h))):
        rows.append((f"{wid}_{i}",float(tvt0),float(_pphy[j]),
                     float(_plgb[j]),float(_pxgb[j])))
har_components=pd.DataFrame(rows,columns=[
    "id","last_known_tvt","har_physics","har_lgb","har_xgb"])
if len(har_components)==0 or not np.isfinite(
        har_components.iloc[:,1:].to_numpy()).all():
    raise RuntimeError("Harshini fresh component inference failed")
print("Harshini fresh components",har_components.shape)

In [ ]:
# Fresh public-weight inference implementation. No saved prediction artifact is read.
"""Strict fresh inference from the Pilkwang public model package.

This script deliberately refuses to read CSV/NPY prediction artifacts. It
builds features from official raw train/test files and loads model weights,
feature schema, and blend configuration only.
"""
from pathlib import Path
import argparse
import importlib.util
import json

import numpy as np
import pandas as pd

ROOT = Path('/kaggle/working')


def load_builder(package):
    p = package/"feature_builders/build_features.py"
    spec = importlib.util.spec_from_file_location("fresh_public_builder", p)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod.build_features


def tree_predictions(frame, package, manifest):
    import joblib
    from catboost import CatBoostRegressor
    import lightgbm as lgb
    import xgboost as xgb
    schema = json.loads((package/"feature_builders/feature_columns.json").read_text())
    out = {}
    for entry in manifest["models"]:
        family = entry["model_family"]
        if family == "sequence_tcn":
            continue
        cols = schema[entry["feature_set"]]
        X = frame[cols].to_numpy(np.float32)
        path = package/entry["path"]
        if family == "xgb":
            model = xgb.XGBRegressor()
            model.load_model(path)
            pred = model.predict(X)
        elif family == "catboost":
            model = CatBoostRegressor()
            model.load_model(path)
            pred = model.predict(X)
        elif family == "hgb":
            # Package was serialized with an older sklearn that exposed the
            # loss extension as top-level ``_loss``.
            import sys
            import sklearn._loss.loss as sklearn_loss
            sys.modules.setdefault("_loss", sklearn_loss)
            model = joblib.load(path)
            pred = model.predict(np.nan_to_num(X))
        elif family == "lgb":
            model = lgb.Booster(model_file=str(path))
            pred = model.predict(X)
        else:
            raise ValueError(f"unsupported family {family}")
        out[entry["prediction_column"]] = np.asarray(pred, np.float32)
    return out


def tcn_prediction(frame, package):
    import torch
    from torch import nn
    ck = torch.load(package/"models/sequence_tcn_tcn_residual.pt",
                    map_location="cpu", weights_only=False)
    cfg = ck["config"]

    class Block(nn.Module):
        def __init__(self, cin, cout, dilation):
            super().__init__()
            pad = dilation*(cfg["kernel_size"]-1)//2
            self.conv1 = nn.Conv1d(cin, cout, cfg["kernel_size"],
                                   padding=pad, dilation=dilation)
            self.conv2 = nn.Conv1d(cout, cout, cfg["kernel_size"],
                                   padding=pad, dilation=dilation)
            self.skip = nn.Conv1d(cin, cout, 1) if cin != cout else nn.Identity()
            self.drop = nn.Dropout(cfg["dropout"])
        def forward(self, x):
            y = self.drop(torch.relu(self.conv1(x)))
            y = self.drop(self.conv2(y))
            return torch.relu(y+self.skip(x))

    class TCN(nn.Module):
        def __init__(self):
            super().__init__()
            blocks = []
            cin = cfg["feature_count"]
            for i in range(cfg["blocks"]):
                blocks.append(Block(cin, cfg["channels"], 2**i))
                cin = cfg["channels"]
            self.net = nn.Sequential(*blocks)
            self.head = nn.Conv1d(cin, 1, 1)
        def forward(self, x):
            return self.head(self.net(x)).squeeze(1)

    model = TCN()
    model.load_state_dict(ck["state_dict"], strict=True)
    model.eval()
    cols = ck["feature_columns"]
    mean = np.asarray(ck["standardizer"]["mean"], np.float32)
    scale = np.asarray(ck["standardizer"]["scale"], np.float32)
    pred = np.empty(len(frame), np.float32)
    # IDs guarantee well grouping and row order; full suffix is observed.
    wells = frame["id"].astype(str).str.rsplit("_", n=1).str[0]
    with torch.inference_mode():
        for _, idx in frame.groupby(wells, sort=False).groups.items():
            pos = frame.index.get_indexer(idx)
            x = frame.iloc[pos][cols].to_numpy(np.float32)
            x = np.nan_to_num((x-mean)/(scale+1e-7))
            xt = torch.from_numpy(x.T[None])
            pred[pos] = model(xt).numpy()[0]
    return pred


def fresh_pilkwang_delta(data, package, sample=None):
    """Return (sample-aligned frame, delta) using weights and raw data only."""
    data, package = Path(data), Path(package)
    manifest = json.loads(
        (package/"metadata/model_package_manifest.json").read_text())
    if sample is None:
        sample = pd.read_csv(data/"sample_submission.csv", dtype={"id": str})
    else:
        sample = sample.copy()
        sample["id"] = sample.id.astype(str)
    frame = load_builder(package)(
        data_dir=data, sample_submission=sample, package_root=package,
        manifest=manifest)
    pred = tree_predictions(frame, package, manifest)
    pred["pred_delta_sequence_tcn_tcn_residual"] = tcn_prediction(frame, package)
    blend = json.loads((package/"stacking/blend_config.json").read_text())
    delta = sum(float(w)*pred[k] for k, w in blend["weights"].items())
    delta = float(blend["postprocess"]["alpha"])*delta
    # Apply documented delta-space well smoothing.
    from scipy.signal import savgol_filter
    wells = frame["id"].astype(str).str.rsplit("_", n=1).str[0]
    for _, idx in frame.groupby(wells, sort=False).groups.items():
        pos = frame.index.get_indexer(idx)
        n = len(pos); win = min(int(blend["postprocess"]["savgol_window"]), n)
        if win % 2 == 0:
            win -= 1
        if win >= 5:
            delta[pos] = savgol_filter(delta[pos], win, 2)
    return frame, np.asarray(delta, float), sorted(pred)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default=str(ROOT/"data"))
    ap.add_argument("--package", default=str(ROOT/"exp/public_artifacts/pilkwang"))
    ap.add_argument("--out", default=str(
        ROOT/"exp/results/public_model_fresh_inference.csv"))
    args = ap.parse_args()
    data, package = Path(args.data), Path(args.package)
    # Explicit denylist: inference cannot silently consume saved predictions.
    forbidden = {"oof", "diagnostics", "submission.csv",
                 "test_base_predictions.csv", "base_test_predictions.npy"}
    assert not any(x in str(package/"feature_builders") for x in forbidden)
    sample = pd.read_csv(data/"sample_submission.csv", dtype={"id": str})
    frame, delta, model_names = fresh_pilkwang_delta(
        data, package, sample=sample)
    last = frame["last_known_TVT"].to_numpy(float)
    tvt = last + delta
    out = sample[["id"]].copy()
    out["tvt"] = tvt
    if len(out) != len(sample) or not np.isfinite(out.tvt).all():
        raise RuntimeError("fresh inference contract failed")
    Path(args.out).parent.mkdir(parents=True, exist_ok=True)
    out.to_csv(args.out, index=False)
    audit = {"rows": len(out), "models": model_names,
             "read_prediction_artifacts": False,
             "hardcoded_test_ids": False,
             "features_built_from_raw_official_files": True,
             "tvt_min": float(out.tvt.min()), "tvt_max": float(out.tvt.max())}
    Path(args.out).with_suffix(".audit.json").write_text(json.dumps(audit, indent=2))
    print(json.dumps(audit, indent=2))




In [ ]:

"""Strict fresh five-leg inference using deployable restricted OOF weights."""
from pathlib import Path as _Path
import importlib.util as _ilu
import json as _json
import numpy as _np
import pandas as _pd
import lightgbm as _lgb

_data=_Path(DATA)
_sample=_pd.read_csv(_data/"sample_submission.csv",dtype={"id":str})
if not _sample.id.is_unique: raise RuntimeError("sample IDs are not unique")

# Pilkwang model-weight inference.
_pil_candidates=[
 _Path("/kaggle/input/datasets/pilkwang/rogii-model-package"),
 _Path("/kaggle/input/rogii-model-package")]
_pil_pkg=next((p for p in _pil_candidates if p.exists()),None)
if _pil_pkg is None: raise RuntimeError("Pilkwang package missing")
_pil_frame,_pil_delta,_pil_models=fresh_pilkwang_delta(
    _data,_pil_pkg,sample=_sample)
if not _pil_frame.id.astype(str).equals(_sample.id):
    raise RuntimeError("Pilkwang order mismatch")

# Stack V4 fresh PF/beam/NCC/spatial features and lgb7 model weights.
_v4_candidates=[
 _Path("/kaggle/input/datasets/boltuzamaki/rogii-stack-v4-lgb7-package"),
 _Path("/kaggle/input/rogii-stack-v4-lgb7-package")]
_v4_pkg=next((p for p in _v4_candidates if p.exists()),None)
if _v4_pkg is None: raise RuntimeError("Stack V4 package missing")
_spec=_ilu.spec_from_file_location("stack_v4_feature_builder",
                                   _v4_pkg/"stack_v4_feature_builder.py")
_v4mod=_ilu.module_from_spec(_spec); _spec.loader.exec_module(_v4mod)
_v4manifest=_json.loads((_v4_pkg/"manifest.json").read_text())
_v4frame=_v4mod.build_inference_features(_data,"test",n_jobs=1)
if _v4manifest["features"] != [c for c in _v4manifest["features"]
                              if c in _v4frame.columns]:
    raise RuntimeError("Stack V4 feature manifest mismatch")
_v4model=_lgb.Booster(model_file=str(_v4_pkg/"stack_v4_lgb7.txt"))
_v4raw=_pd.DataFrame({"id":_v4frame.id.astype(str),
                      "v4_lgb7":_v4model.predict(
                          _v4frame[_v4manifest["features"]])})

# Identity alignment is always sample-left, never positional except after proof.
_all=_sample[["id"]].merge(har_components.assign(
    id=har_components.id.astype(str)),on="id",how="left",validate="one_to_one")
_all=_all.merge(_v4raw,on="id",how="left",validate="one_to_one")
if len(_all)!=len(_sample) or _all.isna().any().any():
    raise RuntimeError("component ID coverage failure")

_W={"har_physics":0.3985925949,"har_lgb":0.2873064512,
    "har_xgb":0.0101013985,"pil_blend":0.2752512537,
    "v4_lgb7":0.2479481257}
_final_delta=(_W["har_physics"]*_all.har_physics.to_numpy()+
              _W["har_lgb"]*_all.har_lgb.to_numpy()+
              _W["har_xgb"]*_all.har_xgb.to_numpy()+
              _W["pil_blend"]*_pil_delta+
              _W["v4_lgb7"]*_all.v4_lgb7.to_numpy())
sub=_sample[["id"]].copy()
sub["tvt"]=_all.last_known_tvt.to_numpy()+_final_delta
if not sub.id.equals(_sample.id): raise RuntimeError("final order mismatch")
if not _np.isfinite(sub.tvt).all(): raise RuntimeError("nonfinite final TVT")
sub.to_csv("/kaggle/working/submission.csv",index=False)
_audit={
 "rows":len(sub),"unique_ids":int(sub.id.nunique()),
 "sample_order_exact":bool(sub.id.equals(_sample.id)),
 "finite":bool(_np.isfinite(sub.tvt).all()),
 "weights":_W,"weight_sum":float(sum(_W.values())),
 "weight_sum_above_one_intentional":True,
 "weight_source":"restricted honest nested GroupKFold CV 8.384425; full OOF positive Ridge alpha=100",
 "fresh_inference":{"harshini":True,"pilkwang":True,"stack_v4_lgb7":True},
 "prediction_csv_inputs_read":False,"hardcoded_test_ids":False,
 "pilkwang_models":_pil_models,
 "tvt_min":float(sub.tvt.min()),"tvt_max":float(sub.tvt.max())}
_Path("/kaggle/working/inference_audit.json").write_text(
    _json.dumps(_audit,indent=2))
print(_json.dumps(_audit,indent=2))
